In [2]:
# import libraries
import ollama
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma



In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
## Lets read the pdf files and split them into chunks
def read_doc(directory):
    file_loader = PyPDFDirectoryLoader(directory)
    documents = file_loader.load()
    return documents

In [5]:
doc = read_doc("documents/")
len(doc)

362

In [6]:
## Devide the docs into chunks
def chunk_data(docs, chunk_size=800, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    doc = text_splitter.split_documents(docs)
    return docs

In [7]:
documents = chunk_data(docs=doc)
len(documents)

362

In [9]:
print("Syncing with Docker Ollama...")
ollama.pull("nomic-embed-text")
ollama.pull("llama3")

embeddings = OllamaEmbeddings(model="nomic-embed-text")
print("✅ Setup Complete")

Syncing with Docker Ollama...
✅ Setup Complete


In [10]:
# --- CONFIGURATION ---
PDF_PATH = "./documents/Vitamin_and_mineral_requirements.pdf"  # Put your PDF file name here
DB_DIR = "./my_local_db"
EMBED_MODEL = "nomic-embed-text"

In [11]:
loader = PyPDFLoader(PDF_PATH)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = loader.load_and_split(text_splitter)

In [12]:
# Create Local Vector DB
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./my_local_db"
)
print(f"✅ Success! {len(chunks)} chunks saved to disk.")

✅ Success! 1154 chunks saved to disk.


In [13]:
query = "Tell me about this document. Whatever you understand." # 👈 Ask anything!

# 1. Search the PDF for relevant text
results = vector_store.similarity_search(query, k=3)
context = "\n\n".join([doc.page_content for doc in results])

# 2. Get answer from Llama 3
response = ollama.chat(model="llama3", messages=[
    {"role": "system", "content": "You are a helpful assistant. Answer the question using ONLY the provided context from the PDF."},
    {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"}
])

print(f"\n🤖 AI ANSWER:\n{response['message']['content']}")



🤖 AI ANSWER:
This appears to be a publication from the World Health Organization (WHO) and the Food and Agriculture Organization of the United Nations. The text does not provide a specific title or topic, but it contains standard disclaimers and acknowledgments common in WHO publications.

The first section explains that permissions for reproduction or translation of WHO publications should be addressed to the Publications department.

The second section contains a disclaimer stating that the designations used and presentation of material do not imply any opinion on the part of the organizations regarding countries, territories, cities, areas, or their authorities. The use of dotted lines on maps indicates approximate border lines where there may not yet be full agreement.

The final section acknowledges the assistance of several individuals in editing and preparing the document.
